# Analisis Dataset Stunting

**Proyek:** Prediksi Stunting pada Balita
**Dataset:** stunting.csv
**Deskripsi:** Dataset berisi data tinggi badan balita berdasarkan umur (bulan) dan jenis kelamin, dengan status stunting (Normal / Stunted / Severely Stunted).

---
## Tahapan Analisis
1. **Setup & Data Loading** - Load dataset dan eksplorasi awal
2. **Data Preprocessing** - Pembersihan data, fixing inkonsistensi
3. **Handling Missing Values** - Deteksi dan penanganan nilai hilang
4. **Outlier Detection & Handling** - Deteksi dan penanganan outlier
5. **Export Clean Dataset** - Simpan dataset bersih dan ringkasan
---

---
# BAGIAN 1: SETUP & DATA LOADING
---

In [ ]:
# ============================================================
# BAGIAN 1: SETUP & DATA LOADING
# Tujuan: Import library, load dataset, dan eksplorasi awal
# ============================================================

# 1.1 Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

# Buat folder processed jika belum ada
os.makedirs('../data/processed', exist_ok=True)

warnings.filterwarnings('ignore')

# Konfigurasi visualisasi
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("Library berhasil diimport.")

In [ ]:
# 1.2 Load Dataset
df_raw = pd.read_csv('../data/raw/stunting.csv', sep=';')

print(f"Dataset berhasil dimuat!")
print(f"Shape: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")

In [ ]:
# 1.3 Eksplorasi Awal
print("=== 5 Data Pertama ===")
display(df_raw.head())

print("\n=== Informasi Dataset ===")
df_raw.info()

print("\n=== Statistik Deskriptif ===")
display(df_raw.describe())

In [ ]:
# 1.4 Distribusi Target (Status)
print("=== Distribusi Status Stunting ===")
status_counts = df_raw['Status'].value_counts()
print(status_counts)
print(f"\nPersentase:")
print(df_raw['Status'].value_counts(normalize=True).mul(100).round(2))

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

status_counts.plot(kind='bar', ax=axes[0], color=['green', 'orange', 'red'])
axes[0].set_title('Distribusi Status Stunting')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Jumlah')
axes[0].tick_params(axis='x', rotation=0)

status_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                   colors=['green', 'orange', 'red'], startangle=90)
axes[1].set_title('Proporsi Status Stunting')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../data/processed/distribusi_status_awal.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nVisualisasi tersimpan: data/processed/distribusi_status_awal.png")

In [ ]:
# 1.5 Export CSV - Data Awal & Statistik
# Simpan data mentah untuk referensi
df_raw.to_csv('../data/processed/stunting_00_raw.csv', index=False)
print("Raw data exported: data/processed/stunting_00_raw.csv")

# Simpan ringkasan statistik
summary_stats = df_raw.describe(include='all').round(2)
summary_stats.to_csv('../data/processed/stunting_00_summary_stats.csv')
print("Summary stats exported: data/processed/stunting_00_summary_stats.csv")

# Simpan distribusi status
dist_status = df_raw['Status'].value_counts().reset_index()
dist_status.columns = ['Status', 'Jumlah']
dist_status['Persentase'] = (dist_status['Jumlah'] / dist_status['Jumlah'].sum() * 100).round(2)
dist_status.to_csv('../data/processed/stunting_00_distribusi_status.csv', index=False)
print("Distribusi status exported: data/processed/stunting_00_distribusi_status.csv")

print(f"\nTotal {df_raw.shape[0]} sampel, {df_raw.shape[1]} kolom siap diproses.")

> **Kesimpulan Bagian 1:** Dataset berhasil dimuat. Terlihat ada 1.351 sampel dengan 4 kolom. Distribusi status menunjukkan ketidakseimbangan kelas (class imbalance) dengan mayoritas Normal.

---
# BAGIAN 2: DATA PREPROCESSING
---

In [ ]:
# ============================================================
# BAGIAN 2: DATA PREPROCESSING
# Tujuan: Membersihkan dataset dari inkonsistensi
# ============================================================

# Buat salinan untuk preprocessing
df = df_raw.copy()

# 2.1 Standardisasi Nama Kolom (lowercase, tanpa spasi)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Nama kolom setelah standardisasi:")
print(df.columns.tolist())

In [ ]:
# 2.2 Cek dan Fix Inkonsistensi String
print("=== Nilai Unik di Kolom 'status' SEBELUM pembersihan ===")
print(df['status'].unique())
print(df['status'].value_counts())

# Fix: hapus spasi berlebih, standardisasi kapitalisasi
df['status'] = df['status'].str.strip().str.title()

print("\n=== Nilai Unik di Kolom 'status' SETELAH pembersihan ===")
print(df['status'].unique())
print(df['status'].value_counts())

In [ ]:
# 2.3 Standardisasi Kolom 'jk' (Jenis Kelamin)
print("=== Nilai Unik di Kolom 'jk' SEBELUM ===")
print(df['jk'].unique())
print(df['jk'].value_counts())

# Standardisasi: L -> Laki-Laki, P -> Perempuan
jk_mapping = {'L': 'Laki-Laki', 'P': 'Perempuan'}
df['jk'] = df['jk'].str.strip().map(jk_mapping)

print("\n=== Nilai Unik di Kolom 'jk' SETELAH ===")
print(df['jk'].unique())
print(df['jk'].value_counts())

In [ ]:
# 2.4 Deteksi Duplikasi Data
print("=== Deteksi Duplikasi Data ===")
duplicates = df.duplicated().sum()
print(f"Jumlah data duplikat: {duplicates}")

if duplicates > 0:
    print("\nBaris duplikat:")
    display(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10))
    # Hapus duplikat
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"\n{duplicates} duplikat telah dihapus.")
    print(f"Shape setelah hapus duplikat: {df.shape}")
else:
    print("Tidak ada data duplikat.")

In [ ]:
# 2.5 Cek dan Konversi Tipe Data
print("=== Tipe Data Sebelum Konversi ===")
print(df.dtypes)

# Konversi umur_bulan dan tinggi ke numerik (jika ada yang object)
df['umur_bulan'] = pd.to_numeric(df['umur_bulan'], errors='coerce')
df['tinggi'] = pd.to_numeric(df['tinggi'], errors='coerce')

print("\n=== Tipe Data Setelah Konversi ===")
print(df.dtypes)

print("\n=== Ringkasan Setelah Preprocessing ===")
display(df.head())
print(f"Shape: {df.shape}")

In [ ]:
# 2.6 Export Intermediate CSV - Hasil Preprocessing
df.to_csv('../data/processed/stunting_01_preprocessed.csv', index=False)
print("Data preprocessing selesai!")
print(f"Intermediate CSV tersimpan: data/processed/stunting_01_preprocessed.csv")
print(f"Shape: {df.shape[0]} baris, {df.shape[1]} kolom")

> **Kesimpulan Bagian 2:** Data preprocessing selesai. Inkonsistensi string telah diperbaiki (spasi pada "Stunted", standardisasi kolom 'jk'). Tipe data sudah sesuai.

---
# BAGIAN 3: HANDLING MISSING VALUES
---

In [ ]:
# ============================================================
# BAGIAN 3: HANDLING MISSING VALUES
# Tujuan: Deteksi dan penanganan missing values
# ============================================================

# 3.1 Deteksi Missing Values
print("=== Jumlah Missing Values per Kolom ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': missing_pct.round(2)})
print(missing_df[missing_df['Jumlah Missing'] > 0] if missing.sum() > 0 else "Tidak ada missing values!")

print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# 3.2 Visualisasi Missing Values
if df.isnull().sum().sum() > 0:
    plt.figure(figsize=(8, 4))
    sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
    plt.title('Peta Missing Values')
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada missing values untuk divisualisasikan. Dataset sudah bersih dari nilai hilang.")

In [ ]:
# 3.3 Penanganan Missing Values (Jika Ada)
total_missing = df.isnull().sum().sum()

if total_missing > 0:
    # Strategi:
    # - Kolom numerik (umur_bulan, tinggi): isi dengan median
    # - Kolom kategorikal (jk, status): isi dengan modus

    for col in df.columns:
        missing_count = df[col].isnull().sum()
        if missing_count > 0:
            if df[col].dtype in ['float64', 'int64']:
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                print(f"Kolom '{col}': {missing_count} missing diisi dengan median ({median_val:.2f})")
            else:
                modus_val = df[col].mode()[0]
                df[col] = df[col].fillna(modus_val)
                print(f"Kolom '{col}': {missing_count} missing diisi dengan modus ({modus_val})")

    print(f"\nTotal {total_missing} missing values telah ditangani.")
    print(f"Missing values setelah penanganan: {df.isnull().sum().sum()}")
else:
    print("Tidak ada missing values. Dataset sudah lengkap.")
    print("Langkah ini tetap dipertahankan untuk mengantisipasi data baru di masa depan.")

# Export Intermediate CSV - Setelah Handling Missing Values
df.to_csv('../data/processed/stunting_02_no_missing.csv', index=False)
print("\nIntermediate CSV tersimpan: data/processed/stunting_02_no_missing.csv")
print(f"Shape: {df.shape[0]} baris, {df.shape[1]} kolom")

> **Kesimpulan Bagian 3:** Dataset stunting tidak memiliki missing values. Semua data lengkap dan siap untuk tahap selanjutnya.

---
# BAGIAN 4: OUTLIER DETECTION & HANDLING
---

In [ ]:
# ============================================================
# BAGIAN 4: OUTLIER DETECTION & HANDLING
# Tujuan: Mendeteksi dan menangani outlier pada kolom numerik
# ============================================================

# 4.1 Statistik Deskriptif Detail
numeric_cols = ['umur_bulan', 'tinggi']

print("=== Statistik Deskriptif Kolom Numerik ===")
display(df[numeric_cols].describe().round(2))

In [ ]:
# 4.2 Visualisasi Distribusi dan Outlier (Boxplot)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot untuk umur_bulan
sns.boxplot(y=df['umur_bulan'], ax=axes[0], color='skyblue')
axes[0].set_title('Boxplot: Umur (Bulan)')
axes[0].set_ylabel('Umur (Bulan)')

# Boxplot untuk tinggi
sns.boxplot(y=df['tinggi'], ax=axes[1], color='lightcoral')
axes[1].set_title('Boxplot: Tinggi Badan (cm)')
axes[1].set_ylabel('Tinggi (cm)')

plt.tight_layout()
plt.show()

# Histogram dengan KDE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['umur_bulan'], kde=True, ax=axes[0], color='skyblue', bins=30)
axes[0].set_title('Distribusi: Umur (Bulan)')
axes[0].set_xlabel('Umur (Bulan)')

sns.histplot(df['tinggi'], kde=True, ax=axes[1], color='lightcoral', bins=30)
axes[1].set_title('Distribusi: Tinggi Badan (cm)')
axes[1].set_xlabel('Tinggi (cm)')

plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Deteksi Outlier dengan Metode IQR
def detect_outliers_iqr(data, column):
    """
    Deteksi outlier menggunakan metode Interquartile Range (IQR).
    Rumus: Q1 - 1.5*IQR hingga Q3 + 1.5*IQR
    """
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]

    return {
        'kolom': column,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'lower_bound': lower_bound,
        'upper_bound': upper_bound,
        'jumlah_outlier': len(outliers),
        'persentase_outlier': round(len(outliers) / len(data) * 100, 2),
        'outliers': outliers
    }

print("=" * 60)
print("DETEKSI OUTLIER DENGAN METODE IQR")
print("=" * 60)

for col in numeric_cols:
    result = detect_outliers_iqr(df, col)
    print(f"\n--- {col.upper()} ---")
    print(f"  Q1 (25%): {result['Q1']:.2f}")
    print(f"  Q3 (75%): {result['Q3']:.2f}")
    print(f"  IQR: {result['IQR']:.2f}")
    print(f"  Batas Bawah: {result['lower_bound']:.2f}")
    print(f"  Batas Atas: {result['upper_bound']:.2f}")
    print(f"  Jumlah Outlier: {result['jumlah_outlier']}")
    print(f"  Persentase Outlier: {result['persentase_outlier']}%")

In [ ]:
# 4.4 Deteksi Outlier dengan Metode Z-Score
try:
    from scipy import stats
    scipy_available = True
except ImportError:
    print("Library scipy belum terinstall. Menginstall sekarang...")
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scipy'])
    from scipy import stats
    scipy_available = True

print("=" * 60)
print("DETEKSI OUTLIER DENGAN METODE Z-SCORE")
print("=" * 60)
print("(Threshold: |Z-score| > 3 dianggap outlier)")

for col in numeric_cols:
    z_scores = np.abs(stats.zscore(df[col]))
    outlier_count = (z_scores > 3).sum()
    outlier_pct = round(outlier_count / len(df) * 100, 2)
    print(f"\n--- {col.upper()} ---")
    print(f"  Jumlah Outlier (Z > 3): {outlier_count}")
    print(f"  Persentase: {outlier_pct}%")

In [ ]:
# 4.5 Hubungan Umur vs Tinggi (Scatter Plot)
plt.figure(figsize=(10, 6))

# Warna berdasarkan status
color_map = {'Normal': 'green', 'Stunted': 'orange', 'Severely Stunted': 'red'}
colors = df['status'].map(color_map)

scatter = plt.scatter(df['umur_bulan'], df['tinggi'], c=colors, alpha=0.6, edgecolors='black', linewidth=0.5)
plt.xlabel('Umur (Bulan)')
plt.ylabel('Tinggi Badan (cm)')
plt.title('Hubungan Umur vs Tinggi Badan (Berdasarkan Status Stunting)')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', label='Normal'),
    Patch(facecolor='orange', label='Stunted'),
    Patch(facecolor='red', label='Severely Stunted')
]
plt.legend(handles=legend_elements, title='Status')

plt.tight_layout()
plt.savefig('../data/processed/scatter_umur_tinggi.png', dpi=100, bbox_inches='tight')
plt.show()
print("Visualisasi tersimpan: data/processed/scatter_umur_tinggi.png")

In [ ]:
# 4.6 Penanganan Outlier
print("=" * 60)
print("PENANGANAN OUTLIER")
print("=" * 60)

# Strategi: Capping (Winsorization) - mengganti outlier dengan batas IQR
# Alasan: Data tinggi dan umur anak memiliki variasi alami,
# namun nilai ekstrim perlu dibatasi agar tidak mengganggu model

df_clean = df.copy()
total_capped = 0

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Hitung outlier sebelum capping
    before_outliers = ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()

    # Lakukan capping
    df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)

    total_capped += before_outliers
    print(f"\n--- {col.upper()} ---")
    print(f"  Batas Bawah: {lower_bound:.2f}")
    print(f"  Batas Atas: {upper_bound:.2f}")
    print(f"  Outlier sebelum capping: {before_outliers}")
    print(f"  Outlier setelah capping: {((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()}")

print(f"\nTotal outlier yang ditangani (capping): {total_capped}")

In [ ]:
# 4.7 Visualisasi Perbandingan Sebelum vs Sesudah Penanganan Outlier
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, col in enumerate(numeric_cols):
    # Sebelum
    sns.boxplot(y=df[col], ax=axes[i, 0], color='lightcoral')
    axes[i, 0].set_title(f'{col.upper()} - SEBELUM Penanganan Outlier')

    # Sesudah
    sns.boxplot(y=df_clean[col], ax=axes[i, 1], color='lightgreen')
    axes[i, 1].set_title(f'{col.upper()} - SETELAH Penanganan Outlier')

plt.tight_layout()
plt.savefig('../data/processed/perbandingan_outlier.png', dpi=100, bbox_inches='tight')
plt.show()
print("Visualisasi tersimpan: data/processed/perbandingan_outlier.png")

In [ ]:
# 4.8 Export Intermediate CSV - Setelah Penanganan Outlier
df_clean.to_csv('../data/processed/stunting_03_after_outlier.csv', index=False)
print("Intermediate CSV tersimpan: data/processed/stunting_03_after_outlier.csv")
print(f"Shape: {df_clean.shape[0]} baris, {df_clean.shape[1]} kolom")

> **Kesimpulan Bagian 4:** Outlier berhasil dideteksi menggunakan metode IQR dan Z-Score. Penanganan dilakukan dengan capping (winsorization) untuk mempertahankan jumlah data sambil mengurangi pengaruh nilai ekstrim.

---
# BAGIAN 5: EXPORT CLEAN DATASET & SUMMARY
---

In [ ]:
# ============================================================
# BAGIAN 5: EXPORT CLEAN DATASET & SUMMARY
# Tujuan: Menyimpan dataset bersih dan merangkum hasil
# ============================================================

# 5.1 Simpan Dataset Bersih
# Simpan dengan delimiter koma untuk kemudahan akses
df_clean.to_csv('../data/processed/stunting_clean.csv', index=False)
print("Dataset bersih tersimpan di: data/processed/stunting_clean.csv")

In [ ]:
# 5.2 Perbandingan Sebelum vs Sesudah Cleaning
print("=" * 60)
print("PERBANDINGAN SEBELUM VS SESUDAH CLEANING")
print("=" * 60)

print(f"\n--- Dimensi Data ---")
print(f"  Sebelum: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")
print(f"  Sesudah: {df_clean.shape[0]} baris, {df_clean.shape[1]} kolom")

print(f"\n--- Missing Values ---")
print(f"  Sebelum: {df_raw.isnull().sum().sum()}")
print(f"  Sesudah: {df_clean.isnull().sum().sum()}")

print(f"\n--- Tipe Data ---")
print(f"  Sebelum:")
print(df_raw.dtypes)
print(f"\n  Sesudah:")
print(df_clean.dtypes)

print(f"\n--- 5 Data Pertama (Sesudah) ---")
display(df_clean.head())

In [ ]:
# 5.3 Statistik Perbandingan Numerik
print("=" * 60)
print("PERBANDINGAN STATISTIK NUMERIK")
print("=" * 60)

for col in numeric_cols:
    print(f"\n--- {col.upper()} ---")
    comparison = pd.DataFrame({
        'Sebelum': df[col].describe(),
        'Sesudah': df_clean[col].describe()
    }).round(2)
    display(comparison)

In [ ]:
# 5.4 Ringkasan Final
print("=" * 60)
print("RINGKASAN FINAL CLEANING DATASET")
print("=" * 60)

data_dict = {"raw_rows": df_raw.shape[0], "clean_rows": df_clean.shape[0], "cols": df_clean.shape[1]}

summary = f"""
Dataset: stunting.csv
Total Sampel Awal: {data_dict['raw_rows']}
Total Sampel Akhir: {data_dict['clean_rows']}
Jumlah Kolom: {data_dict['cols']}

TAHAPAN YANG TELAH DILAKUKAN:
1. Data Loading & Eksplorasi Awal
2. Data Preprocessing:
   - Standardisasi nama kolom
   - Fix inkonsistensi nilai Status (spasi, kapitalisasi)
   - Standardisasi kolom JK (L/P -> Laki-Laki/Perempuan)
   - Deteksi dan hapus duplikasi data (22 duplikat dihapus)
   - Konversi tipe data numerik

3. Handling Missing Values:
   - Tidak ditemukan missing values dalam dataset
   - Handler tetap disiapkan untuk antisipasi data baru

4. Outlier Detection & Handling:
   - Metode IQR (Q1 - 1.5*IQR, Q3 + 1.5*IQR): 1 outlier di tinggi
   - Metode Z-Score (|Z| > 3): 2 outlier di tinggi
   - Penanganan: Capping/Winsorization

5. Export Dataset Bersih ke CSV
"""

print(summary)

# Simpan ringkasan ke file
with open('../data/processed/ringkasan_cleaning.txt', 'w') as f:
    f.write(summary)
print("Ringkasan tersimpan di: data/processed/ringkasan_cleaning.txt")

In [ ]:
# 5.5 Verifikasi Data Bersih
print("=" * 60)
print("VERIFIKASI AKHIR")
print("=" * 60)

# Load kembali dataset yang telah disimpan
df_verified = pd.read_csv('../data/processed/stunting_clean.csv')

print(f"\nDataset berhasil dimuat kembali: {df_verified.shape[0]} baris, {df_verified.shape[1]} kolom")
print(f"\n=== Cek Missing Values ===")
print(df_verified.isnull().sum())

print(f"\n=== Cek Duplikasi ===")
print(f"Jumlah duplikat: {df_verified.duplicated().sum()}")

print(f"\n=== Distribusi Status ===")
print(df_verified['status'].value_counts())

print("\nDataset STUNTING_CLEAN.csv siap digunakan untuk pemodelan!")

---
# BAGIAN 6: MODEL COMPARISON - 8 ALGORITMA KLASIFIKASI
---

In [ ]:
# ============================================================
# BAGIAN 6: MODEL COMPARISON
# Tujuan: Membandingkan 8 algoritma klasifikasi untuk prediksi stunting
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Library imported.")
print("Dataset bersih akan digunakan untuk pemodelan.")

In [ ]:
# 6.1 Load Data & Feature Engineering
df_model = pd.read_csv('../data/processed/stunting_clean.csv')
print(f"Data loaded: {df_model.shape[0]} baris, {df_model.shape[1]} kolom")
print(f"\nDistribusi target (status):")
print(df_model['status'].value_counts())

# Encode JK: Laki-Laki=1, Perempuan=0
df_model['jk_encoded'] = df_model['jk'].map({'Laki-Laki': 1, 'Perempuan': 0})

# Encode target: Normal=0, Stunted=1, Severely Stunted=2
target_mapping = {'Normal': 0, 'Stunted': 1, 'Severely Stunted': 2}
df_model['status_encoded'] = df_model['status'].map(target_mapping)

feature_cols = ['umur_bulan', 'tinggi', 'jk_encoded']
X = df_model[feature_cols].values
y = df_model['status_encoded'].values

print(f"Features: {feature_cols}")
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Kelas target: {np.unique(y)}")

In [ ]:
# 6.2 Split Data & Handle Imbalance dengan SMOTE
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Split 80:20 dengan stratifikasi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print("Distribusi kelas di TRAIN sebelum SMOTE:")
print(pd.Series(y_train).value_counts().sort_index())

# SMOTE untuk mengatasi imbalance
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print("Setelah SMOTE:")
print(f"Train size: {X_train_res.shape[0]}")
print(pd.Series(y_train_res).value_counts().sort_index())

In [ ]:
# 6.3 Definisi 8 Algoritma Klasifikasi
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    '1. Naive Bayes': GaussianNB(),
    '2. Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
    '3. Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, multi_class='ovr'),
    '4. KNN': KNeighborsClassifier(n_neighbors=5),
    '5. SVM': SVC(random_state=42, probability=True),
    '6. Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    '7. XGBoost': XGBClassifier(random_state=42, n_estimators=100, eval_metric='mlogloss'),
    '8. Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100),
}

print(f"8 algoritma siap dilatih: {list(models.keys())}")

In [ ]:
# 6.4 Training & Evaluasi Semua Model
results = []
print("=" * 80)
print(f"{'Algoritma':25} {'Accuracy':10} {'Precision':10} {'Recall':10} {'F1-Score':10}")
print("=" * 80)

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    results.append({
        'Algorithm': name.replace('1. ', '').replace('2. ', '').replace('3. ', '').replace('4. ', '').replace('5. ', '').replace('6. ', '').replace('7. ', '').replace('8. ', ''),
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4)
    })
    print(f"{name:25} {acc:.4f}{'':6} {prec:.4f}{'':6} {rec:.4f}{'':6} {f1:.4f}")

print("=" * 80)

In [ ]:
# 6.5 Export Hasil Perbandingan ke CSV
df_results = pd.DataFrame(results)
df_results = df_results.sort_values('F1-Score', ascending=False).reset_index(drop=True)
df_results.to_csv('../data/processed/model_comparison_results.csv', index=False)

print("Hasil perbandingan model:")
print(df_results.to_string(index=False))
print(f"\nCSV tersimpan: data/processed/model_comparison_results.csv")

In [ ]:
# 6.6 Visualisasi Perbandingan Model (Bar Chart + Tabel)
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Perbandingan 8 Algoritma Klasifikasi - Prediksi Stunting', fontsize=16, fontweight='bold')

# Bar Chart
ax1 = axes[0]
x_pos = np.arange(len(df_results))
width = 0.2
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

for i, (metric, color) in enumerate(zip(['Accuracy', 'Precision', 'Recall', 'F1-Score'], colors)):
    bars = ax1.bar(x_pos + i*width, df_results[metric].values, width, label=metric, color=color, alpha=0.85, edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, df_results[metric].values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{val:.3f}',
                ha='center', va='bottom', fontsize=7, rotation=45)

ax1.set_xticks(x_pos + width*1.5)
ax1.set_xticklabels(df_results['Algorithm'].values, rotation=30, ha='right', fontsize=10)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Perbandingan Metrik per Algoritma', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 1.1)
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# Table
ax2 = axes[1]
ax2.axis('off')
table_data = []
col_labels = ['Rank', 'Algorithm', 'Accuracy', 'Precision', 'Recall', 'F1-Score']
for i, (_, row) in enumerate(df_results.iterrows()):
    table_data.append([
        f'{i+1}',
        row['Algorithm'],
        f'{row["Accuracy"]:.4f}',
        f'{row["Precision"]:.4f}',
        f'{row["Recall"]:.4f}',
        f'{row["F1-Score"]:.4f}'
    ])

table = ax2.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.6)

for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_facecolor('#2E86AB')
        cell.set_text_props(color='white', fontweight='bold')
    elif i == 1:
        cell.set_facecolor('#d4edda')
        cell.set_text_props(fontweight='bold')
    elif i % 2 == 0:
        cell.set_facecolor('#f8f9fa')
    else:
        cell.set_facecolor('white')

ax2.set_title('Ranking Algoritma (by F1-Score)', fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('../data/processed/model_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart tersimpan: data/processed/model_comparison_chart.png")

In [ ]:
# 6.7 Model Terbaik
best = df_results.iloc[0]
print("=" * 60)
print(f"MODEL TERBAIK: {best['Algorithm']}")
print("=" * 60)
print(f"  Accuracy:  {best['Accuracy']:.4f}")
print(f"  Precision: {best['Precision']:.4f}")
print(f"  Recall:    {best['Recall']:.4f}")
print(f"  F1-Score:  {best['F1-Score']:.4f}")
print("=" * 60)

> **Kesimpulan Bagian 6:** Dari 8 algoritma yang diuji, **Random Forest** menunjukkan performa terbaik dengan F1-Score 0.9546, diikuti Gradient Boosting (0.9533) dan Decision Tree (0.9483). Naive Bayes memiliki performa terendah (F1-Score 0.5505) karena asumsi independensi fitur tidak terpenuhi pada data ini.

---
## Kesimpulan Akhir

Proses **Data Cleaning & Model Comparison** pada dataset stunting telah selesai dilakukan dengan tahapan:

| Tahap | Keterangan | Status |
|-------|-----------|--------|
| 1 | Setup & Data Loading | Selesai |
| 2 | Data Preprocessing | Selesai |
| 3 | Handling Missing Values | Selesai |
| 4 | Outlier Detection & Handling | Selesai |
| 5 | Export Clean Dataset | Selesai |
| 6 | Model Comparison (8 Algoritma) | Selesai |

**Hasil Akhir:** Dataset bersih tersimpan di data/processed/stunting_clean.csv

**Best Model:** Random Forest (F1-Score: 0.9546)

---
*Notebook ini dibuat untuk proyek penelitian prediksi stunting pada balita.*